# Day 6：LLaMA 预训练实践 — 从零训练 + 带 KV Cache 的文本生成

> **目标**：在小规模语料上完成 LLaMA 预训练的完整流程——使用 Day 3 手写的 LLaMA 模型，完成数据准备、训练循环、Loss 曲线监控；实现带 KV Cache 的自回归生成，对比有无 Cache 的速度差异。

---

## 总体流程

```
Part 1: 数据准备
  加载语料 → Tokenize → 创建 DataLoader

Part 2: 模型配置
  使用 Day 3 手写的 LLaMA（缩小配置适配单卡训练）

Part 3: 训练循环
  AdamW + Cosine LR + Gradient Clipping → 训练 + 验证

Part 4: 带 KV Cache 的文本生成
  实现 Prefill + Decode 两阶段推理 → 对比不同训练阶段的生成效果

Part 5: 速度对比
  有 Cache vs 无 Cache 的推理速度基准测试
```

In [1]:
import math
import os
import time
from dataclasses import dataclass
from typing import Optional, Tuple, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4090


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

## Part 1：数据准备

使用 TinyShakespeare 语料（与第三周相同），约 1MB 的莎士比亚全集文本。

**与 Week3 的区别**：这次使用 SentencePiece BPE tokenizer（更接近 LLaMA 的真实 tokenizer），而非字符级别。

为了简便，我们这里退回到字符级 tokenizer 以避免额外依赖。实际 LLaMA 使用 SentencePiece（32K 词表）。

In [ ]:
# 加载数据
data_path = 'tinyshakespeare.txt'

if not os.path.exists(data_path):
    import urllib.request
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    urllib.request.urlretrieve(url, data_path)
    print(f"Downloaded {data_path}")

with open(data_path, 'r') as f:
    text = f.read()

print(f"Total characters: {len(text):,}")
print(f"First 200 chars:\n{text[:200]}")

In [ ]:
# 字符级 Tokenizer
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocab size: {vocab_size}")

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

# 编码全部文本
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data shape: {data.shape}")
print(f"Sample encoding: '{text[:20]}' -> {data[:20].tolist()}")

In [ ]:
# 划分训练集和验证集（90/10）
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Train: {len(train_data):,} tokens")
print(f"Val:   {len(val_data):,} tokens")

In [ ]:
class TextDataset(Dataset):
    """滑动窗口式文本数据集。"""
    
    def __init__(self, data: torch.Tensor, block_size: int):
        self.data = data
        self.block_size = block_size
    
    def __len__(self):
        return len(self.data) - self.block_size
    
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y


block_size = 256  # 上下文窗口大小
batch_size = 64

train_dataset = TextDataset(train_data, block_size)
val_dataset = TextDataset(val_data, block_size)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

print(f"Block size: {block_size}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# 测试一个 batch
x_batch, y_batch = next(iter(train_loader))
print(f"\nBatch shapes: x={x_batch.shape}, y={y_batch.shape}")

## Part 2：LLaMA 模型定义

使用 Day 3 手写的 LLaMA 模型，缩小配置以适配单卡训练。

**关键改进**：本次模型加入了 KV Cache 支持。

In [ ]:
@dataclass
class LLaMAConfig:
    dim: int = 512
    n_layers: int = 8
    n_heads: int = 8
    n_kv_heads: int = 4         # GQA: 4 KV heads for 8 Q heads
    vocab_size: int = 65        # 字符级词表
    max_seq_len: int = 256
    norm_eps: float = 1e-5

    @property
    def head_dim(self):
        return self.dim // self.n_heads

    @property
    def ffn_dim(self):
        hidden = int(2 * self.dim * 4 / 3)
        return ((hidden + 255) // 256) * 256

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

In [ ]:
def precompute_freqs_cis(dim: int, seq_len: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(seq_len)
    freqs = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(freqs), freqs)


def apply_rotary_emb(xq, xk, freqs_cis):
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.unsqueeze(0).unsqueeze(2)
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(-2)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(-2)
    return xq_out.type_as(xq), xk_out.type_as(xk)


def repeat_kv(x, n_rep):
    if n_rep == 1:
        return x
    B, T, n_kv_heads, head_dim = x.shape
    x = x[:, :, :, None, :].expand(B, T, n_kv_heads, n_rep, head_dim)
    return x.reshape(B, T, n_kv_heads * n_rep, head_dim)

In [ ]:
class Attention(nn.Module):
    """Grouped Query Attention with KV Cache support."""
    
    def __init__(self, config: LLaMAConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.n_rep = config.n_heads // config.n_kv_heads
        self.head_dim = config.head_dim
        
        self.wq = nn.Linear(config.dim, config.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(config.dim, config.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(config.dim, config.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(config.n_heads * self.head_dim, config.dim, bias=False)
        
        self.cache_k = None
        self.cache_v = None
    
    def forward(self, x, freqs_cis, mask=None, use_cache=False):
        B, T, _ = x.shape
        
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim)
        
        q, k = apply_rotary_emb(q, k, freqs_cis)
        
        if use_cache:
            if self.cache_k is None:
                self.cache_k = k
                self.cache_v = v
            else:
                self.cache_k = torch.cat([self.cache_k, k], dim=1)
                self.cache_v = torch.cat([self.cache_v, v], dim=1)
            k = self.cache_k
            v = self.cache_v
        
        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)
        
        q = q.transpose(1, 2)  # (B, n_heads, T_q, head_dim)
        k = k.transpose(1, 2)  # (B, n_heads, T_kv, head_dim)
        v = v.transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores + mask
        
        attn = F.softmax(scores, dim=-1)
        output = torch.matmul(attn, v)
        
        output = output.transpose(1, 2).contiguous().view(B, -1, self.n_heads * self.head_dim)
        return self.wo(output)
    
    def clear_cache(self):
        self.cache_k = None
        self.cache_v = None

In [ ]:
class SwiGLUFFN(nn.Module):
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.w_gate = nn.Linear(dim, hidden_dim, bias=False)
        self.w_up = nn.Linear(dim, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, dim, bias=False)
    
    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class LLaMABlock(nn.Module):
    def __init__(self, config: LLaMAConfig):
        super().__init__()
        self.attention_norm = RMSNorm(config.dim, config.norm_eps)
        self.attention = Attention(config)
        self.ffn_norm = RMSNorm(config.dim, config.norm_eps)
        self.feed_forward = SwiGLUFFN(config.dim, config.ffn_dim)
    
    def forward(self, x, freqs_cis, mask=None, use_cache=False):
        x = x + self.attention(self.attention_norm(x), freqs_cis, mask, use_cache)
        x = x + self.feed_forward(self.ffn_norm(x))
        return x

In [ ]:
class LLaMA(nn.Module):
    def __init__(self, config: LLaMAConfig):
        super().__init__()
        self.config = config
        
        self.tok_emb = nn.Embedding(config.vocab_size, config.dim)
        self.layers = nn.ModuleList([LLaMABlock(config) for _ in range(config.n_layers)])
        self.norm = RMSNorm(config.dim, config.norm_eps)
        self.lm_head = nn.Linear(config.dim, config.vocab_size, bias=False)
        
        self.freqs_cis = precompute_freqs_cis(config.head_dim, config.max_seq_len * 2)
        
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, tokens, freqs_cis=None, use_cache=False):
        B, T = tokens.shape
        h = self.tok_emb(tokens)
        
        if freqs_cis is None:
            freqs_cis = self.freqs_cis[:T]
        freqs_cis = freqs_cis.to(h.device)
        
        if use_cache and self.layers[0].attention.cache_k is not None:
            mask = None  # Decode 阶段不需要 causal mask（Q 只有 1 个 token）
        else:
            mask = torch.triu(torch.full((T, T), float('-inf'), device=h.device), diagonal=1)
            mask = mask.unsqueeze(0).unsqueeze(0)  # (1, 1, T, T)
        
        for layer in self.layers:
            h = layer(h, freqs_cis, mask, use_cache)
        
        h = self.norm(h)
        logits = self.lm_head(h)
        return logits
    
    def clear_cache(self):
        for layer in self.layers:
            layer.attention.clear_cache()
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())

In [ ]:
# 创建模型
config = LLaMAConfig()
config.vocab_size = vocab_size

model = LLaMA(config).to(device)

total_params = model.count_parameters()
print(f"Model config:")
print(f"  dim={config.dim}, layers={config.n_layers}, heads={config.n_heads}, kv_heads={config.n_kv_heads}")
print(f"  ffn_dim={config.ffn_dim}, max_seq_len={config.max_seq_len}")
print(f"  vocab_size={config.vocab_size}")
print(f"\nTotal parameters: {total_params:,} ({total_params/1e6:.1f}M)")

# 测试前向传播
with torch.no_grad():
    test_input = torch.randint(0, vocab_size, (2, 32)).to(device)
    test_output = model(test_input)
    print(f"\nTest forward pass: input {test_input.shape} → output {test_output.shape}")

## Part 3：训练循环

训练配置对比：

| 配置 | LLaMA-7B (原始) | 我们的小 LLaMA |
|------|----------------|---------------|
| 参数量 | 6.7B | ~30M |
| 优化器 | AdamW | AdamW |
| 学习率 | 3e-4 | 3e-4 |
| LR Schedule | Cosine | Cosine |
| Warmup | 2000 steps | 100 steps |
| 权重衰减 | 0.1 | 0.1 |
| 梯度裁剪 | 1.0 | 1.0 |
| 精度 | BF16 | FP32 |

In [ ]:
# 训练超参数
max_epochs = 10
learning_rate = 3e-4
weight_decay = 0.1
warmup_steps = 100
grad_clip = 1.0
eval_interval = 200      # 每 200 步评估一次
generate_interval = 500  # 每 500 步生成一次样本

# 优化器
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=(0.9, 0.95),
    weight_decay=weight_decay
)

total_steps = max_epochs * len(train_loader)
print(f"Total training steps: {total_steps:,}")
print(f"Steps per epoch: {len(train_loader)}")

In [ ]:
def get_lr(step, warmup_steps, total_steps, max_lr, min_lr=0.0):
    """Cosine learning rate schedule with warmup."""
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= total_steps:
        return min_lr
    decay_ratio = (step - warmup_steps) / (total_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)


@torch.no_grad()
def evaluate(model, val_loader, device, max_batches=50):
    """在验证集上评估 loss。"""
    model.eval()
    losses = []
    for i, (x, y) in enumerate(val_loader):
        if i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

In [ ]:
@torch.no_grad()
def generate_text(model, prompt_str, max_new_tokens=200, temperature=0.8, top_k=40):
    """使用 KV Cache 生成文本。"""
    model.eval()
    model.clear_cache()
    
    tokens = torch.tensor([encode(prompt_str)], dtype=torch.long, device=device)
    T = tokens.shape[1]
    
    # Prefill
    freqs_cis = model.freqs_cis[:T]
    logits = model(tokens, freqs_cis=freqs_cis, use_cache=True)
    
    generated = list(tokens[0].cpu().numpy())
    
    # 从最后一个位置采样
    logits_last = logits[0, -1, :] / temperature
    if top_k > 0:
        topk_vals, _ = torch.topk(logits_last, top_k)
        logits_last[logits_last < topk_vals[-1]] = float('-inf')
    probs = F.softmax(logits_last, dim=-1)
    next_token = torch.multinomial(probs, num_samples=1)
    generated.append(next_token.item())
    
    # Decode
    for step in range(max_new_tokens - 1):
        cur_pos = T + step + 1
        x = next_token.unsqueeze(0)  # (1, 1)
        freqs_cis = model.freqs_cis[cur_pos - 1:cur_pos]
        logits = model(x, freqs_cis=freqs_cis, use_cache=True)
        
        logits_last = logits[0, -1, :] / temperature
        if top_k > 0:
            topk_vals, _ = torch.topk(logits_last, top_k)
            logits_last[logits_last < topk_vals[-1]] = float('-inf')
        probs = F.softmax(logits_last, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        generated.append(next_token.item())
    
    model.clear_cache()
    model.train()
    return decode(generated)

In [ ]:
# 训练前生成（随机输出）
print("=" * 60)
print("训练前生成（应该是随机字符）:")
print("=" * 60)
print(generate_text(model, "ROMEO:", max_new_tokens=200))
print("=" * 60)

In [ ]:
# ===== 主训练循环 =====
train_losses = []
val_losses = []
lr_history = []
step = 0
best_val_loss = float('inf')

print(f"Starting training: {max_epochs} epochs, {total_steps} total steps")
print("=" * 70)

start_time = time.time()

for epoch in range(max_epochs):
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        
        # 更新学习率
        lr = get_lr(step, warmup_steps, total_steps, learning_rate)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
        lr_history.append(lr)
        
        # 前向传播
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        
        # 梯度裁剪
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        
        optimizer.step()
        
        train_losses.append(loss.item())
        
        # 评估
        if step > 0 and step % eval_interval == 0:
            val_loss = evaluate(model, val_loader, device)
            val_losses.append((step, val_loss))
            
            elapsed = time.time() - start_time
            tokens_per_sec = (step + 1) * batch_size * block_size / elapsed
            
            print(f"Step {step:5d} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss:.4f} | "
                  f"LR: {lr:.2e} | Grad Norm: {grad_norm:.2f} | {tokens_per_sec:.0f} tok/s")
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
        
        # 生成样本
        if step > 0 and step % generate_interval == 0:
            print(f"\n--- Generation at step {step} ---")
            sample = generate_text(model, "ROMEO:", max_new_tokens=150, temperature=0.8)
            print(sample[:300])
            print("---\n")
        
        step += 1

total_time = time.time() - start_time
print(f"\nTraining complete! Total time: {total_time:.1f}s")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Final training loss: {train_losses[-1]:.4f}")

In [ ]:
# 绘制 Loss 曲线
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Train Loss
smoothed = np.convolve(train_losses, np.ones(50)/50, mode='valid')
axes[0].plot(smoothed, 'b-', alpha=0.8, linewidth=1)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Train Loss')
axes[0].set_title('Training Loss (smoothed)')
axes[0].grid(True, alpha=0.3)

# Val Loss
val_steps, val_loss_vals = zip(*val_losses) if val_losses else ([], [])
axes[1].plot(val_steps, val_loss_vals, 'ro-', markersize=4)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Val Loss')
axes[1].set_title('Validation Loss')
axes[1].grid(True, alpha=0.3)

# Learning Rate
axes[2].plot(lr_history, 'g-', linewidth=1)
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('llama_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Initial loss (random): {train_losses[0]:.4f} (expected: -ln(1/{vocab_size}) ≈ {math.log(vocab_size):.4f})")
print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Best val loss: {best_val_loss:.4f}")

## Part 4：训练后生成效果

对比不同 prompt 下的生成效果。一个训练良好的模型应该能生成有基本英语结构和莎士比亚风格的文本。

In [ ]:
prompts = [
    "ROMEO:",
    "To be or not to be",
    "KING HENRY:",
    "The ",
    "First Citizen:\n"
]

for prompt in prompts:
    print("=" * 60)
    print(f"Prompt: '{prompt}'")
    print("-" * 60)
    text = generate_text(model, prompt, max_new_tokens=200, temperature=0.8, top_k=40)
    print(text)
    print()

In [ ]:
# Temperature 对比
print("Temperature 对生成多样性的影响：")
print("=" * 70)

for temp in [0.3, 0.8, 1.2]:
    print(f"\n--- Temperature = {temp} ---")
    text = generate_text(model, "ROMEO:", max_new_tokens=150, temperature=temp, top_k=40)
    print(text[:250])
    print()

## Part 5：KV Cache 速度对比

对比有无 KV Cache 的推理速度，验证 Cache 的加速效果。

In [ ]:
@torch.no_grad()
def generate_no_cache(model, prompt_tokens, max_new_tokens=100):
    """不使用 KV Cache 的朴素生成（每步重新计算全序列）。"""
    model.eval()
    tokens = prompt_tokens.clone()
    
    for _ in range(max_new_tokens):
        logits = model(tokens, use_cache=False)  # 每次输入完整序列
        next_token = logits[:, -1:, :].argmax(dim=-1)
        tokens = torch.cat([tokens, next_token], dim=1)
        
        if tokens.shape[1] >= model.config.max_seq_len:
            break
    
    return tokens


@torch.no_grad()
def generate_with_cache(model, prompt_tokens, max_new_tokens=100):
    """使用 KV Cache 的高效生成。"""
    model.eval()
    model.clear_cache()
    
    T = prompt_tokens.shape[1]
    
    # Prefill
    freqs_cis = model.freqs_cis[:T]
    logits = model(prompt_tokens, freqs_cis=freqs_cis, use_cache=True)
    next_token = logits[:, -1:, :].argmax(dim=-1)
    
    generated = [next_token]
    
    # Decode
    for step in range(max_new_tokens - 1):
        cur_pos = T + step + 1
        freqs_cis = model.freqs_cis[cur_pos - 1:cur_pos]
        logits = model(next_token, freqs_cis=freqs_cis, use_cache=True)
        next_token = logits[:, -1:, :].argmax(dim=-1)
        generated.append(next_token)
    
    model.clear_cache()
    all_tokens = torch.cat([prompt_tokens] + generated, dim=1)
    return all_tokens

In [ ]:
# 速度基准测试
prompt = torch.tensor([encode("ROMEO:")], dtype=torch.long, device=device)

gen_lengths = [32, 64, 128]
results = []

for gen_len in gen_lengths:
    # 预热
    _ = generate_with_cache(model, prompt, max_new_tokens=10)
    model.clear_cache()
    _ = generate_no_cache(model, prompt, max_new_tokens=10)
    
    if device == 'cuda':
        torch.cuda.synchronize()
    
    # 无 Cache
    start = time.time()
    for _ in range(3):
        _ = generate_no_cache(model, prompt, max_new_tokens=gen_len)
    if device == 'cuda':
        torch.cuda.synchronize()
    time_no_cache = (time.time() - start) / 3
    
    # 有 Cache
    start = time.time()
    for _ in range(3):
        _ = generate_with_cache(model, prompt, max_new_tokens=gen_len)
    if device == 'cuda':
        torch.cuda.synchronize()
    time_with_cache = (time.time() - start) / 3
    
    speedup = time_no_cache / time_with_cache
    results.append((gen_len, time_no_cache, time_with_cache, speedup))
    
    print(f"Gen length={gen_len:3d} | No Cache: {time_no_cache:.3f}s ({gen_len/time_no_cache:.0f} tok/s) | "
          f"With Cache: {time_with_cache:.3f}s ({gen_len/time_with_cache:.0f} tok/s) | "
          f"Speedup: {speedup:.1f}x")

In [ ]:
# 可视化速度对比
if results:
    gen_lens = [r[0] for r in results]
    no_cache_times = [r[1] for r in results]
    cache_times = [r[2] for r in results]
    speedups = [r[3] for r in results]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(gen_lens, no_cache_times, 'ro-', label='No Cache', linewidth=2)
    axes[0].plot(gen_lens, cache_times, 'bs-', label='With Cache', linewidth=2)
    axes[0].set_xlabel('Generation Length (tokens)')
    axes[0].set_ylabel('Time (seconds)')
    axes[0].set_title('Inference Time: Cache vs No Cache')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].bar(range(len(gen_lens)), speedups, tick_label=[str(g) for g in gen_lens], color='green', alpha=0.7)
    axes[1].set_xlabel('Generation Length (tokens)')
    axes[1].set_ylabel('Speedup (x)')
    axes[1].set_title('KV Cache Speedup')
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('kv_cache_speedup.png', dpi=150, bbox_inches='tight')
    plt.show()

## Part 6：KV Cache 显存分析

验证 Day 5 学到的 KV Cache 显存公式。

In [ ]:
def analyze_kv_cache_memory(config, seq_len, batch_size=1, dtype_bytes=4):
    """分析 KV Cache 的显存占用。"""
    cache_per_token_per_layer = 2 * config.n_kv_heads * config.head_dim * dtype_bytes
    cache_per_token = cache_per_token_per_layer * config.n_layers
    total_cache = cache_per_token * seq_len * batch_size
    
    model_params = sum(p.numel() for p in model.parameters())
    model_memory = model_params * dtype_bytes
    
    print(f"=== KV Cache Memory Analysis ===")
    print(f"Model: dim={config.dim}, layers={config.n_layers}, heads={config.n_heads}, kv_heads={config.n_kv_heads}")
    print(f"Sequence length: {seq_len}, Batch size: {batch_size}, Dtype: {'FP32' if dtype_bytes==4 else 'FP16'}")
    print(f"")
    print(f"Cache per token per layer: {cache_per_token_per_layer:,} bytes ({cache_per_token_per_layer/1024:.1f} KB)")
    print(f"Cache per token (all layers): {cache_per_token:,} bytes ({cache_per_token/1024:.1f} KB)")
    print(f"Total KV Cache: {total_cache:,} bytes ({total_cache/1024/1024:.2f} MB)")
    print(f"Model weights: {model_memory:,} bytes ({model_memory/1024/1024:.2f} MB)")
    print(f"Cache / Model ratio: {total_cache/model_memory:.2%}")
    
    # 如果是 MHA 会怎样？
    mha_total = 2 * config.n_heads * config.head_dim * dtype_bytes * config.n_layers * seq_len * batch_size
    print(f"\n如果使用 MHA (n_kv_heads={config.n_heads}):")
    print(f"  MHA Cache: {mha_total/1024/1024:.2f} MB")
    print(f"  GQA 节省: {(1 - total_cache/mha_total):.0%}")


analyze_kv_cache_memory(config, seq_len=256, batch_size=1)

In [ ]:
# 对比不同规模下的 Cache 占用
print("\n" + "=" * 70)
print("不同 LLaMA 配置的 KV Cache 分析（FP16, seq_len=2048, batch=1）")
print("=" * 70)

configs = [
    ("LLaMA-7B (MHA)",   32, 32, 32, 128, 2),
    ("LLaMA-13B (MHA)",  40, 40, 40, 128, 2),
    ("LLaMA-2 70B (GQA)", 80, 64, 8,  128, 2),
    ("LLaMA-2 70B (if MHA)", 80, 64, 64, 128, 2),
]

seq_len = 2048

print(f"{'Model':<25} {'Layers':>6} {'Heads':>6} {'KV Heads':>8} {'Cache (MB)':>12} {'Cache (GB, BS=32)':>18}")
print("-" * 80)

for name, n_layers, n_heads, n_kv_heads, head_dim, dtype_bytes in configs:
    cache = 2 * n_layers * n_kv_heads * head_dim * seq_len * dtype_bytes
    cache_mb = cache / 1024 / 1024
    cache_gb_bs32 = cache * 32 / 1024 / 1024 / 1024
    print(f"{name:<25} {n_layers:>6} {n_heads:>6} {n_kv_heads:>8} {cache_mb:>12.1f} {cache_gb_bs32:>18.1f}")

## Part 7：回顾与总结

### 本次实践的关键收获

1. **LLaMA 预训练流程** = GPT 预训练流程 + 架构改进（RMSNorm / RoPE / SwiGLU / GQA）
2. **KV Cache 推理** 显著加速自回归生成，加速比随生成长度线性增长
3. **GQA** 在几乎不损失质量的情况下，大幅减少 KV Cache 显存占用
4. **Decode 阶段是 memory-bound**：瓶颈是读取 KV Cache 和模型权重，而非计算

### LLaMA vs GPT 预训练的实践差异

| 维度 | Week 3 GPT 预训练 | Week 4 LLaMA 预训练 |
|------|------------------|--------------------|
| 归一化 | LayerNorm | RMSNorm |
| 位置编码 | 可学习绝对编码（加在 embedding） | RoPE（在 Attention 中旋转 Q/K） |
| FFN | GELU + 2 层 | SwiGLU + 3 层 |
| 注意力 | MHA | GQA（可退化为 MHA） |
| Bias | 有 | 无 |
| 推理 | 朴素生成 | KV Cache 加速生成 |

### 下一步

- Day 7 将精读 LLaMA-2 论文，理解更大规模训练和 RLHF 对齐
- Week 5 将在 LLaMA 基础上做指令微调（Alpaca）

In [ ]:
# 最终生成展示
print("\n" + "=" * 60)
print("LLaMA 预训练模型最终生成效果")
print("=" * 60)

final_text = generate_text(model, "JULIET:\nO Romeo, Romeo! ", max_new_tokens=300, temperature=0.7, top_k=40)
print(final_text)